# 02 · Chunk selection and log-mel cache

CPU is enough (no GPU quota). Inputs: `birdclef-2025` and the aux dataset, which must now also contain `folds.csv` (from `01_eda`).

For every included recording: decode at 32 kHz → split into 5 s chunks (last one zero-padded) → keep up to `MAX_CHUNKS` chunks with the highest RMS, in time order → 128-bin log-mel.

Outputs (`/kaggle/working`):
- `chunks.csv` — one row per chunk: `row` (index into `mel.f16`), `filename`, `start_s`, `rms`. **03 uses the same chunks.**
- `mel.f16` — float16 array `(n_chunks, 128, 313)` written as raw bytes; open with `np.memmap`.
- `mel_config.json`

Attach this notebook's output as an input to `04_cnn`.

In [ ]:
import json, os
from pathlib import Path

import numpy as np
import pandas as pd

ON_KAGGLE = Path('/kaggle/input').exists()
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
INPUT = Path('/kaggle/input') if ON_KAGGLE else REPO

def find_dir(marker):
    hits = sorted(INPUT.rglob(marker))
    assert hits, f'{marker} not found under {INPUT}'
    return hits[0].parent

COMP = find_dir('taxonomy.csv')
AUX = find_dir('folds.csv')
OUT = Path('/kaggle/working') if ON_KAGGLE else REPO / 'cache' / 'features'
OUT.mkdir(parents=True, exist_ok=True)

CFG = dict(sr=32000, chunk_s=5, max_chunks=5, n_mels=128, n_fft=2048, hop=512, fmin=50, fmax=14000, top_db=80)
CHUNK = CFG['sr'] * CFG['chunk_s']
N_FRAMES = CHUNK // CFG['hop'] + 1
folds = pd.read_csv(AUX / 'folds.csv')
print(COMP, AUX, len(folds), 'recordings', '| frames per chunk:', N_FRAMES)

In [ ]:
import soundfile as sf
import torch, torchaudio

def load(fn):
    y, sr = sf.read(str(COMP / 'train_audio' / fn), dtype='float32', always_2d=True)
    y = torch.from_numpy(y.mean(axis=1))
    if sr != CFG['sr']:
        y = torchaudio.functional.resample(y, sr, CFG['sr'])
    return y

def select_chunks(y):
    n = max(1, int(np.ceil(len(y) / CHUNK)))
    y = torch.nn.functional.pad(y, (0, n * CHUNK - len(y)))
    chunks = y.view(n, CHUNK)
    rms = chunks.pow(2).mean(dim=1).sqrt()
    keep = torch.sort(torch.topk(rms, min(n, CFG['max_chunks'])).indices).values
    return chunks[keep], keep.numpy() * CFG['chunk_s'], rms[keep].numpy()

_mel = None
def logmel(chunks):
    global _mel
    if _mel is None:
        torch.set_num_threads(1)
        _mel = torch.nn.Sequential(
            torchaudio.transforms.MelSpectrogram(sample_rate=CFG['sr'], n_fft=CFG['n_fft'], hop_length=CFG['hop'],
                                                 n_mels=CFG['n_mels'], f_min=CFG['fmin'], f_max=CFG['fmax']),
            torchaudio.transforms.AmplitudeToDB('power', top_db=CFG['top_db']))
    with torch.no_grad():
        return _mel(chunks).numpy().astype(np.float16)

def process(fn):
    try:
        chunks, starts, rms = select_chunks(load(fn))
        return fn, logmel(chunks), starts, rms, None
    except Exception as e:
        return fn, None, None, None, repr(e)

# smoke test on one file
fn0 = folds['filename'].iloc[0]
_, m, st, r, err = process(fn0)
assert err is None, err
print(fn0, m.shape, st, r.round(4))
assert m.shape[1:] == (CFG['n_mels'], N_FRAMES)

In [ ]:
from multiprocessing import Pool
from tqdm.auto import tqdm

mel_path = OUT / 'mel.f16'
mel_path.unlink(missing_ok=True)
rows, errors = [], []
with open(mel_path, 'ab') as fh, Pool(os.cpu_count()) as pool:
    for fn, m, starts, rms, err in tqdm(pool.imap(process, folds['filename'], chunksize=8), total=len(folds)):
        if err:
            errors.append((fn, err)); continue
        fh.write(m.tobytes())
        for s, e in zip(starts, rms):
            rows.append((len(rows), fn, float(s), float(e)))

chunks = pd.DataFrame(rows, columns=['row', 'filename', 'start_s', 'rms'])
chunks.to_csv(OUT / 'chunks.csv', index=False)
(OUT / 'mel_config.json').write_text(json.dumps({**CFG, 'n_frames': int(N_FRAMES), 'n_chunks': len(chunks)}, indent=2))
print('chunks:', len(chunks), '| errors:', len(errors), errors[:5])
print('file size GB:', round(mel_path.stat().st_size / 1e9, 2))

In [ ]:
mel = np.memmap(mel_path, dtype=np.float16, mode='r', shape=(len(chunks), CFG['n_mels'], N_FRAMES))
assert mel_path.stat().st_size == mel.size * 2
print('chunks per recording:', chunks.groupby('filename').size().value_counts().sort_index().to_dict())
print('recordings without chunks:', len(set(folds['filename']) - set(chunks['filename'])))
sample = mel[np.random.default_rng(0).choice(len(chunks), 500, replace=False)].astype(np.float32)
print('dB range over 500 chunks:', sample.min().round(1), sample.max().round(1))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
for a, i in zip(ax, [0, len(chunks) // 2, len(chunks) - 1]):
    a.imshow(mel[i].astype(np.float32), origin='lower', aspect='auto'); a.set_title(chunks.loc[i, 'filename'], fontsize=8)
plt.tight_layout(); plt.show()